In [ ]:
# If needed, uncomment:
# !pip install -q pandas numpy matplotlib seaborn scipy

import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import chisquare

warnings.filterwarnings("ignore")

# ================================================================
# 1. LOAD DATA
# ================================================================
file_path = "Table1.csv"

try:
    df = pd.read_csv(file_path, low_memory=False)
    print(f"File loaded successfully: {file_path}")
    print(f"Rows: {df.shape[0]} | Columns: {df.shape[1]}")
except FileNotFoundError:
    print(f"Error: file {file_path} was not found.")
    raise

# ================================================================
# 2. FIND DATE COLUMN
# ================================================================
date_cols = [c for c in df.columns if "data" in c.lower() or "timestamp" in c.lower() or "date" in c.lower()]

if not date_cols:
    print("ERROR: No date column was found.")
else:
    target_date_col = date_cols[0]
    df["Visit_Date"] = pd.to_datetime(df[target_date_col], errors="coerce")
    df_ev = df.dropna(subset=["Visit_Date"]).copy()

    if len(df_ev) == 0:
        print("ERROR: The detected date values could not be parsed correctly.")
    else:
        df_ev["Month"] = df_ev["Visit_Date"].dt.month

        # ========================================================
        # 3. IDENTIFY RESPIRATORY DISEASES
        # ========================================================
        respiratory_terms = r"\b(asma|dpoc|bronquite|doença pulmonar obstrutiva)\b"
        text_cols = [c for c in df_ev.columns if df_ev[c].dtype == object]

        cond_text = df_ev[text_cols].apply(
            lambda col: col.astype(str).str.contains(respiratory_terms, case=False, na=False, regex=True)
        ).any(axis=1)

        respiratory_struct_cols = [
            c for c in df_ev.columns
            if ("asma" in c.lower() or "dpoc" in c.lower())
            and "choice=" in c.lower()
            and "não" not in c.lower()
        ]

        if respiratory_struct_cols:
            cond_struct = df_ev[respiratory_struct_cols].apply(
                lambda col: col.astype(str).str.lower().str.strip().isin(
                    ["checked", "1", "1.0", "sim", "verdadeiro"]
                )
            ).any(axis=1)

            df_ev["Respiratory_Case"] = (cond_text | cond_struct).astype(int)
        else:
            df_ev["Respiratory_Case"] = cond_text.astype(int)

        # ========================================================
        # 4. MONTHLY AGGREGATION + SÃO PAULO CLIMATE DATA
        # ========================================================
        df_monthly = df_ev.groupby("Month").agg(
            Total_Visits=("Record ID", "count"),
            Respiratory_Cases=("Respiratory_Case", "sum")
        ).reset_index()

        df_monthly["Respiratory_Rate_Percent"] = (
            df_monthly["Respiratory_Cases"] / df_monthly["Total_Visits"] * 100
        )

        sao_paulo_temp = {
            1: 23.1, 2: 23.5, 3: 22.5, 4: 21.2, 5: 18.4, 6: 17.5,
            7: 16.6, 8: 17.7, 9: 19.3, 10: 20.5, 11: 21.6, 12: 22.4
        }

        month_names = {
            1: "Jan", 2: "Feb", 3: "Mar", 4: "Apr", 5: "May", 6: "Jun",
            7: "Jul", 8: "Aug", 9: "Sep", 10: "Oct", 11: "Nov", 12: "Dec"
        }

        all_months = pd.DataFrame({"Month": range(1, 13)})
        df_monthly = all_months.merge(df_monthly, on="Month", how="left").fillna(0)
        df_monthly["Month_Name"] = df_monthly["Month"].map(month_names)
        df_monthly["Average_Temp_SP"] = df_monthly["Month"].map(sao_paulo_temp)

        # ========================================================
        # 5. STATISTICAL TEST (CHI-SQUARE)
        # ========================================================
        observed_cases = df_monthly["Respiratory_Cases"].values
        total_visits = df_monthly["Total_Visits"].values

        chi2_stat, p_val = np.nan, np.nan

        if total_visits.sum() > 0 and observed_cases.sum() > 0:
            global_rate = observed_cases.sum() / total_visits.sum()
            expected_cases = total_visits * global_rate

            mask = total_visits > 0
            if mask.sum() > 1:
                chi2_stat, p_val = chisquare(f_obs=observed_cases[mask], f_exp=expected_cases[mask])

                if p_val < 0.05:
                    significance = f"Statistically Significant (p = {p_val:.4f})"
                    title_color = "#c0392b"
                else:
                    significance = f"Not Significant (p = {p_val:.4f})"
                    title_color = "#7f8c8d"
            else:
                significance = "Insufficient data for statistical test"
                title_color = "#000000"
        else:
            significance = "No respiratory case data available"
            title_color = "#000000"

        # ========================================================
        # 6. EXPORT TABLES (CSVs)
        # ========================================================
        seasonal_table = df_monthly[
            ["Month", "Month_Name", "Total_Visits", "Respiratory_Cases", "Respiratory_Rate_Percent", "Average_Temp_SP"]
        ].copy()
        seasonal_table.columns = [
            "Month (Numeric)",
            "Month",
            "Total Visits (N)",
            "Respiratory Cases (N)",
            "Case Rate (%)",
            "Average Temperature in São Paulo (°C)",
        ]
        seasonal_table["Case Rate (%)"] = seasonal_table["Case Rate (%)"].round(2)
        seasonal_table_file = "respiratory_seasonality_base_en.csv"
        seasonal_table.to_csv(seasonal_table_file, index=False, encoding="utf-8-sig")

        stats_table = pd.DataFrame({
            "Metric": [
                "Total Visits Analyzed",
                "Total Respiratory Cases (Asthma/COPD/Bronchitis)",
                "Chi-Square Statistic (X²)",
                "P-Value (Significance)",
                "Seasonality Interpretation",
            ],
            "Value": [
                total_visits.sum(),
                observed_cases.sum(),
                round(chi2_stat, 4) if not np.isnan(chi2_stat) else "N/A",
                f"{p_val:.4e}" if not np.isnan(p_val) else "N/A",
                significance.split(" (")[0],
            ],
        })
        stats_table_file = "seasonality_statistics_base_en.csv"
        stats_table.to_csv(stats_table_file, index=False, encoding="utf-8-sig")

        print(f"\n{'=' * 80}")
        print("DATA TABLES EXPORTED SUCCESSFULLY:")
        print(f"- '{seasonal_table_file}' (month-by-month: N, rates, and temperature)")
        print(f"- '{stats_table_file}' (chi-square test results)")
        print(f"{'=' * 80}")

        # ========================================================
        # 7. CLIMATE-CLINICAL VISUALIZATION
        # ========================================================
        sns.set_theme(style="whitegrid")
        fig, ax1 = plt.subplots(figsize=(14, 8))

        bar_color = "#3498db"
        sns.barplot(
            data=df_monthly,
            x="Month_Name",
            y="Respiratory_Rate_Percent",
            color=bar_color,
            alpha=0.8,
            ax=ax1
        )
        ax1.set_ylabel(
            "% of Respiratory Cases in the Month (Asthma/COPD)",
            color=bar_color,
            fontsize=13,
            fontweight="bold",
        )
        ax1.tick_params(axis="y", labelcolor=bar_color)
        ax1.set_xlabel("Month of the Year", fontsize=13, fontweight="bold")
        ax1.grid(False)

        ax2 = ax1.twinx()
        line_color = "#e74c3c"
        sns.lineplot(
            data=df_monthly,
            x="Month_Name",
            y="Average_Temp_SP",
            color=line_color,
            marker="o",
            linewidth=3,
            markersize=10,
            ax=ax2,
        )
        ax2.set_ylabel(
            "Average Temperature in São Paulo (°C)",
            color=line_color,
            fontsize=13,
            fontweight="bold",
        )
        ax2.tick_params(axis="y", labelcolor=line_color)
        ax2.set_ylim(bottom=0, top=30)

        plt.title(
            f"Seasonality: Respiratory Diseases vs. Temperature (São Paulo)\nSeasonal Variance: {significance}",
            fontsize=16,
            fontweight="bold",
            pad=20,
            color=title_color,
        )

        for p in ax1.patches:
            if p.get_height() > 0:
                ax1.annotate(
                    f"{p.get_height():.1f}%",
                    (p.get_x() + p.get_width() / 2.0, p.get_height()),
                    ha="center",
                    va="bottom",
                    xytext=(0, 5),
                    textcoords="offset points",
                    fontweight="bold",
                    color="#2c3e50",
                )

        fig.tight_layout()
        plt.show()

        # ========================================================
        # 8. SEASONAL IMPACT REPORT IN CONSOLE
        # ========================================================
        print(f"\n{'=' * 80}")
        print("EPIDEMIOLOGICAL SEASONALITY SUMMARY")
        print(f"{'=' * 80}")
        print(f" -> Visits analyzed (with valid date): {total_visits.sum()}")
        print(f" -> Total REAL episodes of Asthma/COPD/Bronchitis: {observed_cases.sum()}")
        print(f" -> Statistical diagnosis: {significance}")

        if not np.isnan(p_val) and p_val < 0.05:
            peak_month = df_monthly.loc[df_monthly["Respiratory_Rate_Percent"].idxmax()]
            print(f"\n[MANAGEMENT ALERT]")
            print(
                f" -> The variation is real. The peak occurred in {peak_month['Month_Name']} "
                f"({peak_month['Respiratory_Rate_Percent']:.1f}% of total demand)."
            )
            print(
                f" -> The average temperature in São Paulo during {peak_month['Month_Name']} "
                f"was {peak_month['Average_Temp_SP']}°C."
            )
        print(f"{'=' * 80}\n")